# PySpark aplicado al retail de cosméticos

Material docente para mostrar cómo PySpark facilita el procesamiento y la validación distribuida usando el dataset **Ecommerce events history in cosmetics shop**. Todo corre en Docker: primero ejecutamos el mismo flujo en modo local y luego repetimos los pasos críticos conectando el notebook a un clúster Spark con dos workers.


## Flujo general
1. Preparar el repositorio y descomprimir el dataset (automático en la próxima celda).
2. **Escenario A (local)**: `SparkSession` con `local[2]` (mismos recursos que un worker), repasamos DataFrames vs. RDDs, validaciones básicas y un pipeline de `pyspark.ml`, registrando el tiempo de cada paso.
3. **Escenario B (cluster)**: cerramos la sesión anterior, apuntamos al master `spark://spark-master:7077`, verificamos los executors y corremos una agregación que reparte el trabajo entre workers para comparar tiempos.

> El contenedor `pyspark-notebook` está limitado a 2 vCPU/3 GB y se instala la extensión `jupyterlab_execute_time`, por lo que verás los tiempos directamente en la UI, además del magic `%%time` en las celdas clave. Antes de la parte distribuida asegúrate de tener `docker compose up` con `spark-master`, `spark-worker-1`, `spark-worker-2` y `pyspark-notebook` activos.


## 0. Preparación de archivos
El zip original debe estar en `data/raw/ecommerce-events-history-in-cosmetics-shop.zip`. Esta celda detecta la carpeta compartida (funciona dentro o fuera de Docker), crea `data/bronze/` y descomprime los CSV si no existen.


In [1]:
from pathlib import Path
import os
import zipfile


def detect_project_dir() -> Path:
    """Busca la ruta compartida para que el driver y los workers vean los mismos archivos."""
    candidates = [
        os.environ.get('SHARED_PROJECT_DIR'),
        '/opt/project',
        os.getcwd(),
    ]
    for raw_path in candidates:
        if not raw_path:
            continue
        path = Path(raw_path).expanduser().resolve()
        if (path / 'data').exists():
            return path
    return Path.cwd().resolve()


PROJECT_DIR = detect_project_dir()
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
BRONZE_DIR = DATA_DIR / 'bronze'

print(f'Usando directorio del proyecto: {PROJECT_DIR}')
zip_path = RAW_DIR / 'ecommerce-events-history-in-cosmetics-shop.zip'
if not zip_path.exists():
    raise FileNotFoundError('Copia el zip de Kaggle en data/raw antes de continuar')

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
if not any(BRONZE_DIR.glob('*.csv')):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(BRONZE_DIR)
    print('Archivos descomprimidos en', BRONZE_DIR)
else:
    print('CSV ya disponibles; se reutilizarán')

csv_files = sorted(BRONZE_DIR.glob('*.csv'))
print(f'Se detectaron {len(csv_files)} archivos de eventos. Ejemplo: {[p.name for p in csv_files[:3]]}')


Usando directorio del proyecto: /opt/project
CSV ya disponibles; se reutilizarán
Se detectaron 5 archivos de eventos. Ejemplo: ['2019-Dec.csv', '2019-Nov.csv', '2019-Oct.csv']


### Dependencias de PySpark que usaremos durante el notebook


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    input_file_name,
    col,
    to_timestamp,
    hour,
    sum as spark_sum,
    when,
    countDistinct,
    approx_count_distinct,
    desc,
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator


## Instrumentación de tiempos
- El contenedor local usa 2 vCPU/3 GB y la sesión `local[2]` para simular un worker individual.
- Las celdas que afectan el tiempo total incluyen el magic `%%time` para registrar la duración en la salida.
- JupyterLab carga la extensión `jupyterlab_execute_time`, por lo que también verás un sello de tiempo sobre cada celda.


## 1. PySpark local (driver y ejecutores en el mismo contenedor)
Limitamos la sesión a `local[2]` para igualar los recursos de un worker y observar cuánto tarda cada paso cuando no hay paralelismo distribuido.


In [3]:
local_spark = (
    SparkSession.builder
    .appName('CosmeticsEventsLocal')
    .master('local[2]')  # limitamos cores para comparar con un worker
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
print('Master usado:', local_spark.sparkContext.master)
print('Parallelism por defecto:', local_spark.sparkContext.defaultParallelism)
local_spark


Master usado: local[2]
Parallelism por defecto: 2


### 1.1 Cargar el dataset y revisar el esquema


In [4]:
events_df = (
    local_spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(BRONZE_DIR / '*.csv'))
    .withColumn('source_file', input_file_name())
)

events_df.cache()
print('Registros totales:', events_df.count())
events_df.show(5, truncate=False)


Registros totales: 20692840
+-------------------+----------------+----------+-------------------+-------------+---------+-----+---------+------------------------------------+--------------------------------------------+
|event_time         |event_type      |product_id|category_id        |category_code|brand    |price|user_id  |user_session                        |source_file                                 |
+-------------------+----------------+----------+-------------------+-------------+---------+-----+---------+------------------------------------+--------------------------------------------+
|2019-12-01 00:00:00|remove_from_cart|5712790   |1487580005268456287|NULL         |f.o.x    |6.27 |576802932|51d85cb0-897f-48d2-918b-ad63965c12dc|file:///opt/project/data/bronze/2019-Dec.csv|
|2019-12-01 00:00:00|view            |5764655   |1487580005411062629|NULL         |cnd      |29.05|412120092|8adff31e-2051-4894-9758-224bfa8aec18|file:///opt/project/data/bronze/2019-Dec.csv|
|2019-12-01 

In [5]:
events_df.printSchema()
events_df.describe(['price']).show()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- source_file: string (nullable = false)

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         20692840|
|   mean| 8.53473548048573|
| stddev|19.38142460687464|
|    min|           -79.37|
|    max|           327.78|
+-------+-----------------+



### 1.2 DataFrames vs. RDDs
Convertimos el DataFrame a un RDD para replicar el ejemplo tradicional y comparar transformaciones.


In [6]:
%%time
events_rdd = events_df.select('event_type').rdd.map(lambda row: row.event_type)
sample_events = events_rdd.take(10)
event_counts = (
    events_rdd
    .map(lambda evt: (evt, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
    .collect()
)
sample_events, event_counts[:5]


CPU times: user 50.1 ms, sys: 13.8 ms, total: 63.9 ms
Wall time: 35.3 s


(['remove_from_cart',
  'view',
  'cart',
  'view',
  'view',
  'view',
  'cart',
  'view',
  'view',
  'cart'],
 [('view', 9657821),
  ('cart', 5768333),
  ('remove_from_cart', 3979679),
  ('purchase', 1287007)])

### 1.3 Reglas ligeras de validación de datos
Detectamos precios inválidos, categorías ausentes y sesiones de compra sin precio para mostrar métricas de calidad.


In [7]:
%%time
validation_row = events_df.select(
    spark_sum(when((col('price').isNull()) | (col('price') <= 0), 1).otherwise(0)).alias('precios_invalidos'),
    spark_sum(when(col('category_code').isNull(), 1).otherwise(0)).alias('categorias_faltantes'),
    (
        countDistinct(when(col('event_type') == 'purchase', col('user_session')))
        - countDistinct(when((col('event_type') == 'purchase') & (col('price') > 0), col('user_session')))
    ).alias('sesiones_compra_sin_precio')
).collect()[0]
validation_row.asDict()


CPU times: user 13.3 ms, sys: 3.69 ms, total: 17 ms
Wall time: 3.05 s


{'precios_invalidos': 104288,
 'categorias_faltantes': 20339246,
 'sesiones_compra_sin_precio': 0}

### 1.4 Pipeline rápido de Machine Learning
Creamos características simples (`price_clean`, `event_hour`) y entrenamos una regresión logística. Se toma una muestra del 20 % para mantener la práctica ligera.


In [8]:
%%time
prepared_df = (
    events_df
    .withColumn('event_ts', to_timestamp('event_time'))
    .withColumn('event_hour', hour('event_ts'))
    .withColumn('price_clean', col('price').cast('double'))
    .withColumn('label', (col('event_type') == 'purchase').cast('int'))
    .select('price_clean', 'event_hour', 'label')
    .dropna()
)
sample_df = prepared_df.sample(withReplacement=False, fraction=0.2, seed=42)
sample_df.count()


CPU times: user 4.59 ms, sys: 1.1 ms, total: 5.69 ms
Wall time: 1.13 s


4143284

In [9]:
%%time
assembler = VectorAssembler(inputCols=['price_clean', 'event_hour'], outputCol='features')
scaler = StandardScaler(withMean=True, withStd=True, inputCol='features', outputCol='scaled_features')
lr = LogisticRegression(featuresCol='scaled_features', labelCol='label')
pipeline = Pipeline(stages=[assembler, scaler, lr])

train_df, test_df = sample_df.randomSplit([0.7, 0.3], seed=7)
model = pipeline.fit(train_df)
predictions = model.transform(test_df)
predictions.select('price_clean', 'event_hour', 'label', 'probability', 'prediction').show(10, truncate=False)

evaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
auc = evaluator.evaluate(predictions)
print(f'AUC (purchase vs. no purchase): {auc:.3f}')


+-----------+----------+-----+----------------------------------------+----------+
|price_clean|event_hour|label|probability                             |prediction|
+-----------+----------+-----+----------------------------------------+----------+
|0.0        |0         |0    |[0.9184796135957876,0.08152038640421244]|0.0       |
|0.0        |1         |0    |[0.9189573947899209,0.08104260521007911]|0.0       |
|0.0        |2         |0    |[0.9194326213959568,0.08056737860404317]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0        |3         |0    |[0.9199053041982198,0.08009469580178019]|0.0       |
|0.0

### 1.5 Cerrar la sesión local
Liberamos recursos antes de conectarnos al clúster.


In [10]:
local_spark.stop()


## 2. PySpark conectado al clúster Dockerizado
Aquí liberamos el trabajo sobre ambos workers (`spark://spark-master:7077`) para comparar tiempos y revisar las métricas de los ejecutores.


In [11]:

cluster_master_url = 'spark://spark-master:7077'
cluster_builder = (
    SparkSession.builder
    .appName('CosmeticsEventsCluster')
    .master(cluster_master_url)
    .config('spark.sql.shuffle.partitions', '16')
)

if os.environ.get('SPARK_DRIVER_HOST'):
    cluster_builder = cluster_builder.config('spark.driver.host', os.environ['SPARK_DRIVER_HOST'])
cluster_builder = cluster_builder.config('spark.driver.bindAddress', os.environ.get('SPARK_DRIVER_BIND_ADDRESS', '0.0.0.0'))

cluster_spark = cluster_builder.getOrCreate()
print('Master usado:', cluster_spark.sparkContext.master)
print('Parallelism por defecto:', cluster_spark.sparkContext.defaultParallelism)
cluster_spark


Master usado: spark://spark-master:7077
Parallelism por defecto: 2


### 2.1 Validar los ejecutores disponibles
Mostramos la memoria reportada por cada executor para comprobar que nos conectamos correctamente al clúster.


In [12]:
%%time
status_scala = cluster_spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
status_java = cluster_spark._jvm.scala.collection.JavaConverters.mapAsJavaMapConverter(status_scala).asJava()
executor_stats = []
for host_port in status_java.keySet():
    if 'driver' in host_port:
        continue
    mem_tuple = status_java.get(host_port)
    executor_stats.append({
        'host_port': host_port,
        'total_bytes': mem_tuple._1(),
        'remaining_bytes': mem_tuple._2(),
    })
print(f'Executores visibles: {executor_stats}')
executor_stats


Executores visibles: [{'host_port': 'pyspark-notebook:40671', 'total_bytes': 1099746508, 'remaining_bytes': 1099746508}]
CPU times: user 8.53 ms, sys: 727 µs, total: 9.26 ms
Wall time: 58.6 ms


[{'host_port': 'pyspark-notebook:40671',
  'total_bytes': 1099746508,
  'remaining_bytes': 1099746508}]

### 2.2 Relectura distribuida del dataset
La ruta es idéntica, pero ahora los CSV se cargan desde los workers gracias al volumen compartido `/opt/project`.


In [13]:
%%time
cluster_events_df = (
    cluster_spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(BRONZE_DIR / '*.csv'))
    .withColumn('source_file', input_file_name())
    .repartition(24, 'source_file')
    .cache()
)

print('Particiones actuales:', cluster_events_df.rdd.getNumPartitions())
print('Total de registros:', cluster_events_df.count())
cluster_events_df.show(5, truncate=False)


Particiones actuales: 24
Total de registros: 20692840
+-------------------+----------------+----------+-------------------+-------------+--------+-----+---------+------------------------------------+--------------------------------------------+
|event_time         |event_type      |product_id|category_id        |category_code|brand   |price|user_id  |user_session                        |source_file                                 |
+-------------------+----------------+----------+-------------------+-------------+--------+-----+---------+------------------------------------+--------------------------------------------+
|2019-11-01 00:00:02|view            |5802432   |1487580009286598681|NULL         |NULL    |0.32 |562076640|09fafd6c-6c99-46b1-834f-33527f4de241|file:///opt/project/data/bronze/2019-Nov.csv|
|2019-11-01 00:00:09|cart            |5844397   |1487580006317032337|NULL         |NULL    |2.38 |553329724|2067216c-31b5-455d-a1cc-af0575a34ffb|file:///opt/project/data/bronze/2019-

### 2.3 Perfilado distribuido (schema y estadísticas)
Repetimos el perfilado para observar cómo se comporta la lectura distribuida e identificar diferencias frente a la sesión local.


In [14]:
%%time
cluster_events_df.printSchema()
cluster_events_df.describe(['price']).show()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- source_file: string (nullable = false)

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          20692840|
|   mean| 8.534735480461539|
| stddev|19.381424606873683|
|    min|            -79.37|
|    max|            327.78|
+-------+------------------+

CPU times: user 2.56 ms, sys: 2.38 ms, total: 4.93 ms
Wall time: 935 ms


### 2.4 DataFrames vs. RDDs en cluster
El flujo es idéntico al local, pero ahora las particiones se procesan en los workers.


In [15]:
%%time
cluster_events_rdd = cluster_events_df.select('event_type').rdd.map(lambda row: row.event_type)
cluster_sample_events = cluster_events_rdd.take(10)
cluster_event_counts = (
    cluster_events_rdd
    .map(lambda evt: (evt, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
    .collect()
)
cluster_sample_events, cluster_event_counts[:5]


CPU times: user 41.8 ms, sys: 14.1 ms, total: 55.9 ms
Wall time: 22.9 s


(['view',
  'cart',
  'view',
  'cart',
  'remove_from_cart',
  'remove_from_cart',
  'view',
  'view',
  'remove_from_cart',
  'view'],
 [('view', 9657821),
  ('cart', 5768333),
  ('remove_from_cart', 3979679),
  ('purchase', 1287007)])

### 2.5 Reglas de validación en cluster
Aplicamos las mismas reglas para confirmar que los conteos se mantienen y para comparar tiempos.


In [16]:
%%time
cluster_validation_row = cluster_events_df.select(
    spark_sum(when((col('price').isNull()) | (col('price') <= 0), 1).otherwise(0)).alias('precios_invalidos'),
    spark_sum(when(col('category_code').isNull(), 1).otherwise(0)).alias('categorias_faltantes'),
    (
        countDistinct(when(col('event_type') == 'purchase', col('user_session')))
        - countDistinct(when((col('event_type') == 'purchase') & (col('price') > 0), col('user_session')))
    ).alias('sesiones_compra_sin_precio')
).collect()[0]
cluster_validation_row.asDict()


CPU times: user 21.2 ms, sys: 4.71 ms, total: 25.9 ms
Wall time: 2.85 s


{'precios_invalidos': 104288,
 'categorias_faltantes': 20339246,
 'sesiones_compra_sin_precio': 0}

### 2.6 Pipeline distribuido de Machine Learning
Entrenamos el mismo pipeline sobre la sesión distribuida para aprovechar los dos workers y contrastar el `%%time`.


In [17]:
%%time
cluster_prepared_df = (
    cluster_events_df
    .withColumn('event_ts', to_timestamp('event_time'))
    .withColumn('event_hour', hour('event_ts'))
    .withColumn('price_clean', col('price').cast('double'))
    .withColumn('label', (col('event_type') == 'purchase').cast('int'))
    .select('price_clean', 'event_hour', 'label')
    .dropna()
)
cluster_sample_df = cluster_prepared_df.sample(withReplacement=False, fraction=0.2, seed=42)
cluster_sample_df.count()


CPU times: user 3.91 ms, sys: 2.27 ms, total: 6.18 ms
Wall time: 1.08 s


4136701

In [18]:
%%time
cluster_assembler = VectorAssembler(inputCols=['price_clean', 'event_hour'], outputCol='features')
cluster_scaler = StandardScaler(withMean=True, withStd=True, inputCol='features', outputCol='scaled_features')
cluster_lr = LogisticRegression(featuresCol='scaled_features', labelCol='label')
cluster_pipeline = Pipeline(stages=[cluster_assembler, cluster_scaler, cluster_lr])

cluster_train_df, cluster_test_df = cluster_sample_df.randomSplit([0.7, 0.3], seed=7)
cluster_model = cluster_pipeline.fit(cluster_train_df)
cluster_predictions = cluster_model.transform(cluster_test_df)
cluster_predictions.select('price_clean', 'event_hour', 'label', 'probability', 'prediction').show(10, truncate=False)

cluster_evaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
cluster_auc = cluster_evaluator.evaluate(cluster_predictions)
print(f'AUC distribuido: {cluster_auc:.3f}')


+-----------+----------+-----+----------------------------------------+----------+
|price_clean|event_hour|label|probability                             |prediction|
+-----------+----------+-----+----------------------------------------+----------+
|-47.62     |13        |1    |[0.7089812943930481,0.29101870560695187]|0.0       |
|0.0        |1         |0    |[0.9197755976648386,0.08022440233516137]|0.0       |
|0.0        |1         |0    |[0.9197755976648386,0.08022440233516137]|0.0       |
|0.0        |1         |0    |[0.9197755976648386,0.08022440233516137]|0.0       |
|0.0        |2         |0    |[0.9202100132270757,0.07978998677292426]|0.0       |
|0.0        |2         |0    |[0.9202100132270757,0.07978998677292426]|0.0       |
|0.0        |3         |0    |[0.9206422793872668,0.07935772061273316]|0.0       |
|0.0        |3         |0    |[0.9206422793872668,0.07935772061273316]|0.0       |
|0.0        |4         |0    |[0.9210724045855453,0.07892759541445471]|0.0       |
|0.0

### 2.7 Agregaciones pesadas para observar el cluster
Agrupamos por `category_id`, contamos compras, sesiones y usuarios únicos. Ejecuta esta celda mientras revisas la UI de Spark (`http://localhost:8080`) y compara el `%%time` reportado vs. la ejecución local para dimensionar la ganancia de paralelismo.


In [19]:
%%time
cluster_agg = (
    cluster_events_df
    .groupBy('category_id')
    .agg(
        spark_sum(when(col('event_type') == 'purchase', 1).otherwise(0)).alias('compras'),
        approx_count_distinct('user_id').alias('usuarios_unicos'),
        countDistinct('user_session').alias('sesiones_unicas')
    )
    .orderBy(desc('compras'))
)
cluster_agg.show(10, truncate=False)
#cluster_agg.explain('formatted')


+-------------------+-------+---------------+---------------+
|category_id        |compras|usuarios_unicos|sesiones_unicas|
+-------------------+-------+---------------+---------------+
|1487580007675986893|80137  |62466          |217715         |
|1487580006317032337|50555  |67629          |170255         |
|1487580005092295511|44870  |130420         |300197         |
|1487580005595612013|44334  |65556          |189223         |
|1487580005671109489|43452  |55884          |149558         |
|1602943681873052386|36216  |95420          |211373         |
|1487580009286598681|33862  |49237          |93297          |
|1487580005268456287|30450  |86158          |188463         |
|1487580005134238553|25702  |37264          |89395          |
|1487580009445982239|24142  |60223          |117497         |
+-------------------+-------+---------------+---------------+
only showing top 10 rows

CPU times: user 17.7 ms, sys: 3.25 ms, total: 20.9 ms
Wall time: 16.9 s


### 2.4 Finalizar la sesión distribuida


In [20]:
cluster_spark.stop()
